# Project 3: AI Recommendation Logic
### Modern Recommender Systems on the Book-Crossing Dataset

This notebook implements a complete end-to-end recommendation system using the **Book-Crossing** dataset. We will explore several classes of recommendation algorithms, from simple popularity metrics to classic collaborative filtering and state-of-the-art deep learning methods.

#### Goals & Objectives:
1. **Exploratory Data Analysis (EDA):** Understand users, books, and rating sparsity.
2. **Popularity-Based Recommendations:** Establishing a robust baseline.
3. **Content-Based Filtering:** Match user-expressed interests (keywords, book titles) using TF-IDF and Cosine Similarity.
4. **Collaborative Filtering:** Use matrix factorization (SVD) on the user-item interaction matrix.
5. **Neural Collaborative Filtering (Deep Learning):** Build and train a PyTorch neural network that learns embeddings for users and items.
6. **Unified Recommendation Interface:** Build an interactive recommender logic function.


## 1. Recommendation System Taxonomy

Before diving into code, let's review the taxonomy of recommender systems to understand where our models fit. The diagram below illustrates how recommendations are categorized: from traditional collaborative and content-based approaches to modern hybrid and deep learning methods.

![Recommendation System Taxonomy](recsys_taxonomy2.png)

### Key Paradigms:
- **Collaborative Filtering (CF):** Recommends items based on the past interactions (ratings, clicks) of users. It operates under the assumption that if User A and User B agree on one issue, they are more likely to agree on others.
- **Content-Based Filtering:** Focuses on the properties of items (e.g., text descriptions, genre, author). It recommends items similar to those a user liked in the past.
- **Hybrid Methods:** Combine collaborative and content-based approaches to leverage the strengths of both.
- **Deep Learning / Advanced AI:** Uses neural networks (such as embeddings, autoencoders, two-tower models) to model non-linear interactions and handle rich context.


## 2. Classic vs. Deep Learning Architectures

We will implement both classic and deep learning architectures to contrast their methodologies:

### A. Classic Recommendation (Collaborative & Content-based)
Classic collaborative filtering often builds a user-item rating matrix. Methods like Matrix Factorization (e.g., Singular Value Decomposition - SVD) decompose this high-dimensional, sparse matrix into low-rank user and item latent factor matrices.

![Classic Recommendation System](classicRec.png)

### B. Deep Learning Recommendation (Neural Collaborative Filtering)
Neural Collaborative Filtering (NCF) replaces the simple dot product of classic Matrix Factorization with a neural network architecture. By concatenating or multiplying user and item embeddings and feeding them through multi-layer perceptrons (MLP), the model learns complex, non-linear relationships.

![Deep Learning Recommendation](DeepRec.png)


## 3. Libraries & Environment Setup


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)
print('All libraries successfully imported!')


All libraries successfully imported!


## 4. Loading and Cleaning the Data


In [2]:
# Let's load the datasets
print('Loading Books.csv...')
books = pd.read_csv('Books.csv', low_memory=False)
print('Loading Ratings.csv...')
ratings = pd.read_csv('Ratings.csv', low_memory=False)
print('Loading Users.csv...')
users = pd.read_csv('Users.csv', low_memory=False)

print('\nData loaded successfully!')
print(f'Books shape: {books.shape}')
print(f'Ratings shape: {ratings.shape}')
print(f'Users shape: {users.shape}')


Loading Books.csv...
Loading Ratings.csv...
Loading Users.csv...

Data loaded successfully!
Books shape: (271360, 8)
Ratings shape: (1149780, 3)
Users shape: (278858, 3)


### Basic Cleaning and Preprocessing


In [3]:
# Inspecting missing values
print('--- Missing values in Books ---')
print(books.isnull().sum())
print('\n--- Missing values in Ratings ---')
print(ratings.isnull().sum())
print('\n--- Missing values in Users ---')
print(users.isnull().sum())

# Handle missing values in Users.csv (Age column)
# Fill age outliers and null values with median
median_age = users['Age'].median()
users['Age'] = users['Age'].apply(lambda x: x if 10 <= x <= 100 else median_age)

# Drop unused image URL columns in Books to save memory
books_cleaned = books.drop(columns=['Image-URL-S', 'Image-URL-M', 'Image-URL-L'], errors='ignore')

# Check for and drop duplicate entries
books_cleaned = books_cleaned.drop_duplicates(subset=['ISBN'])
ratings = ratings.drop_duplicates()
users = users.drop_duplicates(subset=['User-ID'])

print('\nPreprocessing completed!')


--- Missing values in Books ---
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

--- Missing values in Ratings ---
User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

--- Missing values in Users ---
User-ID          0
Location         0
Age         110762
dtype: int64

Preprocessing completed!


## 5. Exploratory Data Analysis (EDA)


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribution of Book Ratings
sns.countplot(x='Book-Rating', data=ratings, ax=axes[0, 0], hue='Book-Rating', legend=False)
axes[0, 0].set_title('Distribution of Book Ratings')
axes[0, 0].set_xlabel('Rating Value')
axes[0, 0].set_ylabel('Count')

# Note: 0 indicates implicit ratings, 1-10 represents explicit feedback

# 2. Age Distribution of Users
sns.histplot(users['Age'], bins=30, kde=True, ax=axes[0, 1], color='teal')
axes[0, 1].set_title('User Age Distribution')
axes[0, 1].set_xlabel('Age')
axes[0, 1].set_ylabel('Density')

# 3. Top 10 Most Rated Books
top_books = ratings['ISBN'].value_counts().head(10).reset_index()
top_books = top_books.merge(books_cleaned, on='ISBN', how='left')
sns.barplot(x='count', y='Book-Title', data=top_books, ax=axes[1, 0], hue='Book-Title', legend=False)
axes[1, 0].set_title('Top 10 Most Rated Books')
axes[1, 0].set_xlabel('Number of Ratings')
axes[1, 0].set_ylabel('Book Title')

# 4. Top Authors by Number of Ratings
top_authors = ratings.merge(books_cleaned, on='ISBN', how='inner')['Book-Author'].value_counts().head(10).reset_index()
sns.barplot(x='count', y='Book-Author', data=top_authors, ax=axes[1, 1], hue='Book-Author', legend=False)
axes[1, 1].set_title('Top 10 Authors by Rating Counts')
axes[1, 1].set_xlabel('Rating Counts')
axes[1, 1].set_ylabel('Author Name')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=100)
plt.show()


<string>:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Matrix Sparsity Analysis
Recommendation systems suffer from the 'cold start' and matrix sparsity problems. Let's calculate the sparsity of our Book-Crossing rating matrix.


In [5]:
unique_users = ratings['User-ID'].nunique()
unique_books = ratings['ISBN'].nunique()
total_possible_ratings = unique_users * unique_books
actual_ratings = len(ratings)
sparsity = (1 - (actual_ratings / total_possible_ratings)) * 100

print(f'Unique Users: {unique_users:,}')
print(f'Unique Books: {unique_books:,}')
print(f'Total Possible Ratings: {total_possible_ratings:,}')
print(f'Actual Ratings Present: {actual_ratings:,}')
print(f'Rating Matrix Sparsity: {sparsity:.6f}%')


Unique Users: 105,283
Unique Books: 340,556
Total Possible Ratings: 35,854,757,348
Actual Ratings Present: 1,149,780
Rating Matrix Sparsity: 99.996793%


## 6. Model 1: Popularity-Based Recommender System

Popularity-based recommendations are useful for cold-starting new users. Simply sorting by average rating can be biased towards items with very few reviews (e.g. a book with one rating of 10 stars).
We use a **Weighted Rating formula** similar to IMDb's:

$$WR = \left(\frac{v}{v+m} \cdot R\right) + \left(\frac{m}{v+m} \cdot C\right)$$

Where:
- $v$ is the number of ratings for the book.
- $m$ is the minimum ratings required to be listed (we use the 90th percentile).
- $R$ is the average rating of the book.
- $C$ is the mean rating across the whole dataset.

We will evaluate this using only **explicit ratings (1-10)** to avoid the bias of implicit ratings (0).


In [6]:
# Filter for explicit ratings
explicit_ratings = ratings[ratings['Book-Rating'] > 0]

# Calculate count and mean rating for each book
book_stats = explicit_ratings.groupby('ISBN').agg(R=('Book-Rating', 'mean'), v=('Book-Rating', 'count')).reset_index()

C = explicit_ratings['Book-Rating'].mean()
m = book_stats['v'].quantile(0.90)  # 90th percentile of ratings count

def weighted_rating(df, m=m, C=C):
    v = df['v']
    R = df['R']
    return (v/(v+m) * R) + (m/(v+m) * C)

# Filter books with ratings >= m
popular_books = book_stats[book_stats['v'] >= m].copy()
popular_books['score'] = popular_books.apply(weighted_rating, axis=1)
popular_books = popular_books.sort_values(by='score', ascending=False)

# Merge with book titles
popular_books = popular_books.merge(books_cleaned, on='ISBN', how='left')

print(f'Global Mean Rating (C): {C:.2f}')
print(f'Minimum Ratings Cutoff (m): {m:.2f}')
print('\n--- Top 5 Popular Books (Weighted Ratings) ---')
print(popular_books[['Book-Title', 'Book-Author', 'v', 'R', 'score']].head(5).to_string(index=False))


Global Mean Rating (C): 7.60
Minimum Ratings Cutoff (m): 4.00

--- Top 5 Popular Books (Weighted Ratings) ---
                                           Book-Title      Book-Author  v        R    score
Harry Potter and the Chamber of Secrets Postcard Book    J. K. Rowling 23 9.869565 9.533491
       The Two Towers (The Lord of the Rings, Part 2) J. R. R. Tolkien 25 9.720000 9.427733
                                      The Giving Tree Shel Silverstein 20 9.750000 9.391844
                         Dilbert: A Book of Postcards      Scott Adams 13 9.923077 9.376721
                                                  NaN              NaN 21 9.714286 9.376171


## 7. Model 2: Content-Based Recommender (Preference Matching)

Content-based filtering matches user input (choices, keywords, or favorite books) with metadata of other books using text vectorization (TF-IDF) and similarity metrics (Cosine Similarity).
We vectorize the combination of **Book-Title, Book-Author, and Publisher** to construct item profiles.


In [7]:
# Since Books.csv is large, we take a subset of the most rated books for similarity calculations to prevent out-of-memory errors
top_rated_isbns = ratings['ISBN'].value_counts().head(20000).index
content_books = books_cleaned[books_cleaned['ISBN'].isin(top_rated_isbns)].copy()
content_books = content_books.fillna('')

# Create content metadata feature
content_books['features'] = content_books['Book-Title'] + ' ' + content_books['Book-Author'] + ' ' + content_books['Publisher']

# Compute TF-IDF matrix
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(content_books['features'])
print(f'TF-IDF Matrix shape: {tfidf_matrix.shape}')

# Simple content recommendation function based on user text input
def get_content_recommendations(user_interest_query, top_n=5):
    # Vectorize the query
    query_vec = tfidf.transform([user_interest_query])
    # Calculate cosine similarity with all books
    sims = cosine_similarity(query_vec, tfidf_matrix).flatten()
    # Get top_n indices
    top_indices = sims.argsort()[-top_n:][::-1]
    results = content_books.iloc[top_indices].copy()
    results['similarity_score'] = sims[top_indices]
    return results[['Book-Title', 'Book-Author', 'Publisher', 'similarity_score']]


TF-IDF Matrix shape: (19058, 10000)


In [8]:
# Test query matching
query = 'Harry Potter magic wizard'
print(f'Recommendations for interest: "{query}"')
print(get_content_recommendations(query, top_n=5).to_string(index=False))


Recommendations for interest: "Harry Potter magic wizard"
                                                      Book-Title     Book-Author              Publisher  similarity_score
                   J. K. Rowling: The Wizard Behind Harry Potter    Marc Shapiro     St. Martin's Press          0.546654
         Harry Potter and the Prisoner of Azkaban (Harry Potter)    J.K. Rowling  Scholastic Paperbacks          0.545745
Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))   J. K. Rowling Arthur A. Levine Books          0.521103
             The Science of Harry Potter: How Magic Really Works Roger Highfield          Penguin Books          0.482896
                                                The Magic Bullet     Harry Stein                 Island          0.432263


## 8. Model 3: Collaborative Filtering via Matrix Factorization (SVD)

Collaborative Filtering relies on the interaction matrix. We use Singular Value Decomposition (SVD) to decompose the matrix and predict missing ratings.
To make the matrix size computationally feasible, we filter for users who have rated at least **50 books** and books that have received at least **50 ratings**.


In [9]:
# Filter to reduce sparsity and dimensionality
user_counts = ratings['User-ID'].value_counts()
book_counts = ratings['ISBN'].value_counts()

ratings_filtered = ratings[
    ratings['User-ID'].isin(user_counts[user_counts >= 50].index) &
    ratings['ISBN'].isin(book_counts[book_counts >= 50].index)
].copy()

print(f'Filtered Ratings shape: {ratings_filtered.shape}')

# Pivot ratings to make User-Item Matrix
pivot_df = ratings_filtered.pivot(index='User-ID', columns='ISBN', values='Book-Rating').fillna(0)
print(f'User-Item Matrix shape: {pivot_df.shape}')

# Perform Singular Value Decomposition (SVD)
# We will use k=15 latent factors
U, sigma, Vt = svds(pivot_df.values, k=15)
sigma = np.diag(sigma)

# Reconstruct the approximated ratings matrix
predicted_ratings = np.dot(np.dot(U, sigma), Vt)
predicted_df = pd.DataFrame(predicted_ratings, columns=pivot_df.columns, index=pivot_df.index)
print('SVD Reconstruction completed!')


Filtered Ratings shape: (143240, 3)
User-Item Matrix shape: (3291, 2185)
SVD Reconstruction completed!


In [10]:
def recommend_books_svd(user_id, num_recommendations=5):
    if user_id not in pivot_df.index:
        return 'User ID not in collaborative filtering training subset. Using popularity-based fallback instead.'
    
    # Get and sort the user's predictions
    user_row_number = pivot_df.index.get_loc(user_id)
    sorted_user_predictions = predicted_df.iloc[user_row_number].sort_values(ascending=False)
    
    # Get books already rated by the user
    user_data = ratings_filtered[ratings_filtered['User-ID'] == user_id]
    rated_isbns = user_data['ISBN'].tolist()
    
    # Recommend books not yet rated
    recommendations = sorted_user_predictions.drop(labels=rated_isbns, errors='ignore').head(num_recommendations).reset_index()
    recommendations.columns = ['ISBN', 'Predicted_Rating']
    recommendations = recommendations.merge(books_cleaned, on='ISBN', how='left')
    return recommendations[['Book-Title', 'Book-Author', 'Predicted_Rating']]

example_user = pivot_df.index[0]
print(f'SVD Recommendations for User ID: {example_user}')
print(recommend_books_svd(example_user).to_string(index=False))


SVD Recommendations for User ID: 243
                                        Book-Title   Book-Author  Predicted_Rating
                                          The Firm  John Grisham          1.065193
                                               NaN           NaN          0.974847
               The Red Tent (Bestselling Backlist) Anita Diamant          0.958825
   Divine Secrets of the Ya-Ya Sisterhood: A Novel Rebecca Wells          0.835734
Where the Heart Is (Oprah's Book Club (Paperback))  Billie Letts          0.719174


## 9. Model 4: Deep Learning Collaborative Filtering (Neural CF)

Now we build a Neural Collaborative Filtering (NCF) model using **PyTorch**. The architecture maps users and books to trainable low-dimensional vector representations (embeddings), concatenates them, and feeds them into fully connected neural network layers to output the predicted rating.


In [11]:
# Prepare data mappings for Embeddings
user_ids = ratings_filtered['User-ID'].unique()
book_isbns = ratings_filtered['ISBN'].unique()

user2idx = {o:i for i,o in enumerate(user_ids)}
book2idx = {o:i for i,o in enumerate(book_isbns)}

ratings_filtered['user_idx'] = ratings_filtered['User-ID'].map(user2idx)
ratings_filtered['book_idx'] = ratings_filtered['ISBN'].map(book2idx)

num_users = len(user2idx)
num_books = len(book2idx)
print(f'Number of mapped users: {num_users}, Number of mapped books: {num_books}')


Number of mapped users: 3291, Number of mapped books: 2185


In [12]:
# Define PyTorch Dataset
class BookRatingDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.books = torch.tensor(df['book_idx'].values, dtype=torch.long)
        # Normalize ratings to range [0, 1] for training stability
        self.ratings = torch.tensor(df['Book-Rating'].values / 10.0, dtype=torch.float32)
        
    def __len__(self):
        return len(self.users)
        
    def __getitem__(self, idx):
        return self.users[idx], self.books[idx], self.ratings[idx]

dataset = BookRatingDataset(ratings_filtered)
train_loader = DataLoader(dataset, batch_size=256, shuffle=True)

# Define NCF Model
class NeuralCollaborativeFiltering(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=32):
        super(NeuralCollaborativeFiltering, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
    def forward(self, user, item):
        u_emb = self.user_embedding(user)
        i_emb = self.item_embedding(item)
        x = torch.cat([u_emb, i_emb], dim=-1)
        output = self.fc(x)
        return output.squeeze(-1)

model = NeuralCollaborativeFiltering(num_users, num_books, embedding_dim=16)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
print('PyTorch Model instantiated successfully!')


PyTorch Model instantiated successfully!


In [13]:
# Train Model for 2 epochs for quick demonstration
epochs = 2
model.train()
for epoch in range(epochs):
    total_loss = 0
    for users_b, books_b, ratings_b in train_loader:
        optimizer.zero_grad()
        predictions = model(users_b, books_b)
        loss = criterion(predictions, ratings_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(users_b)
    print(f'Epoch {epoch+1}/{epochs} - Train MSE Loss: {total_loss / len(dataset):.4f}')


Epoch 1/2 - Train MSE Loss: 0.1304
Epoch 2/2 - Train MSE Loss: 0.1149


In [14]:
# Prediction and Deep Recommendation Function
def recommend_books_deep(user_id, num_recommendations=5):
    if user_id not in user2idx:
        return 'User ID not in Deep Learning training subset. Fallback to popularity recommended.'
    
    user_idx = user2idx[user_id]
    model.eval()
    
    # Get books already rated
    rated_isbns = ratings_filtered[ratings_filtered['User-ID'] == user_id]['ISBN'].tolist()
    unrated_isbns = [isbn for isbn in book_isbns if isbn not in rated_isbns]
    unrated_indices = [book2idx[isbn] for isbn in unrated_isbns]
    
    # Feed to neural net
    user_tensor = torch.tensor([user_idx] * len(unrated_indices), dtype=torch.long)
    item_tensor = torch.tensor(unrated_indices, dtype=torch.long)
    
    with torch.no_grad():
        preds = model(user_tensor, item_tensor).detach().numpy() * 10.0 # Denormalize back to rating scale
    
    top_idx = preds.argsort()[-num_recommendations:][::-1]
    recommended_isbns = [unrated_isbns[i] for i in top_idx]
    scores = preds[top_idx]
    
    recs_df = pd.DataFrame({'ISBN': recommended_isbns, 'Predicted_Rating': scores})
    recs_df = recs_df.merge(books_cleaned, on='ISBN', how='left')
    return recs_df[['Book-Title', 'Book-Author', 'Predicted_Rating']]

print(f'Deep Learning (NCF) Recommendations for User ID: {example_user}')
print(recommend_books_deep(example_user).to_string(index=False))


Deep Learning (NCF) Recommendations for User ID: 243
                                                            Book-Title       Book-Author  Predicted_Rating
                      Harry Potter and the Chamber of Secrets (Book 2)     J. K. Rowling          3.513849
                                                              Watchers    Dean R. Koontz          3.442267
Ice Bound: A Doctor's Incredible Battle for Survival at the South Pole Dr. Jerri Nielsen          3.434420
                                                    Survivor : A Novel   CHUCK PALAHNIUK          3.433203
                    Harry Potter and the Order of the Phoenix (Book 5)     J. K. Rowling          3.431851


## 10. Unified Recommender Interface

Let's bundle all algorithms under a unified interface `recommend_books()` that allows choice of model and handles fallbacks gracefully.


In [15]:
def recommend_books(user_id=None, query=None, method='popularity', top_n=5):
    """
    Universal recommender router.
    Methods: 'popularity', 'content', 'svd', 'deep'
    """
    print(f'=== RECOMMENDATION SERVICE (Method: {method}) ===\n')
    if method == 'popularity':
        return popular_books[['Book-Title', 'Book-Author', 'score']].head(top_n)
    elif method == 'content':
        if not query:
            return 'Please provide a search query for content-based recommendations.'
        return get_content_recommendations(query, top_n)
    elif method == 'svd':
        if not user_id:
            return 'Please provide a User ID for Collaborative Filtering.'
        return recommend_books_svd(user_id, top_n)
    elif method == 'deep':
        if not user_id:
            return 'Please provide a User ID for Neural Collaborative Filtering.'
        return recommend_books_deep(user_id, top_n)
    else:
        return 'Invalid method chosen.'


In [16]:
# Showcase popular books
print(recommend_books(method='popularity', top_n=3).to_string(index=False))


=== RECOMMENDATION SERVICE (Method: popularity) ===

                                           Book-Title      Book-Author    score
Harry Potter and the Chamber of Secrets Postcard Book    J. K. Rowling 9.533491
       The Two Towers (The Lord of the Rings, Part 2) J. R. R. Tolkien 9.427733
                                      The Giving Tree Shel Silverstein 9.391844


In [17]:
# Showcase content search
print(recommend_books(query='Science fiction space travel stars', method='content', top_n=3).to_string(index=False))


=== RECOMMENDATION SERVICE (Method: content) ===

                                                                                                       Book-Title       Book-Author                          Publisher  similarity_score
                                                                                                          N-Space       Larry Niven                Tor Science Fiction          0.539141
                                                                              Harvest of Stars (Harvest of Stars)     Poul Anderson                Tor Science Fiction          0.486056
The Science Fiction Hall of Fame: The Greatest Science Fiction Stories of All Time (Science Fiction Hall of Fame) Robert Silverberg Harper Mass Market Paperbacks (Mm)          0.397457


In [18]:
# Showcase SVD
print(recommend_books(user_id=example_user, method='svd', top_n=3).to_string(index=False))


=== RECOMMENDATION SERVICE (Method: svd) ===

                         Book-Title   Book-Author  Predicted_Rating
                           The Firm  John Grisham          1.065193
                                NaN           NaN          0.974847
The Red Tent (Bestselling Backlist) Anita Diamant          0.958825


In [19]:
# Showcase Deep learning NCF
print(recommend_books(user_id=example_user, method='deep', top_n=3).to_string(index=False))


=== RECOMMENDATION SERVICE (Method: deep) ===

                                                            Book-Title       Book-Author  Predicted_Rating
                      Harry Potter and the Chamber of Secrets (Book 2)     J. K. Rowling          3.513849
                                                              Watchers    Dean R. Koontz          3.442267
Ice Bound: A Doctor's Incredible Battle for Survival at the South Pole Dr. Jerri Nielsen          3.434420
